<a href="https://colab.research.google.com/github/BhavyeNijhawan/swarkavach/blob/main/notebooks/04_fusion_and_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04. Fusion and the evaluation grid

This is where the project's central claim gets tested. The claim is not "late
fusion helps". The claim is that the two branches fail in different places,
that the cross-modal features see something neither branch can compute alone,
and that this shows up specifically in the two off-diagonal cells of the
four-cell grid: a human scammer (the audio branch has nothing to find) and a
legitimate robocall (the audio branch fires on a synthetic voice that is not
committing a crime).

The four arms come straight from `schema.ABLATION_ARMS`, so each arm sees
exactly the feature group it is entitled to and nothing else:

| arm | features it sees |
| --- | --- |
| `audio_only` | the 8 `voice` features |
| `text_only` | the 9 `intent` features |
| `late_fusion` | both of the above, 17 features |
| `full` | all 25, including the 8 `cross` features |

`late_fusion` against `full` is the measurement that matters. If they are the
same, the cross-modal block is decoration and the report should say so.

**Needs:** no GPU strictly, but a GPU makes the RawNet re-scoring of the codec
sweep about ten times faster. A T4 is worth using if you have quota left.

**Time:** 10 to 20 minutes.

**Reads:** `data/results/antispoof_branch_scores.json` from notebook 02 and
`data/results/intent_branch_scores.json` from notebook 03, plus
`data/models/antispoof_rawnet.pt`. Any of these can be missing; the notebook
falls back to the locally trained branch and marks the export accordingly.

**Writes back:** `ablation_results.json`, `calibration.json`,
`robustness_results.json`, `ttd_results.json`.

In [ ]:
# ---------------------------------------------------------------------------
# Get the repository and install the Colab stack.
# Re-running this cell is safe: if the clone is already on disk it pulls.
# ---------------------------------------------------------------------------

# CHANGE THIS to your own fork before you run anything.
REPO_URL = "https://github.com/BhavyeNijhawan/swarkavach.git"
REPO_DIR = "swarkavach"
BRANCH = "main"

import os
import sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = BASE / REPO_DIR

if (REPO / "pyproject.toml").exists():
    print(f"{REPO} is already cloned, pulling instead")
    !git -C "{REPO}" pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("cwd:", os.getcwd())

!pip install -q -r requirements-colab.txt
!pip install -q -e .

import swarkavach
print("swarkavach", swarkavach.__version__, "ready")

## Bring in what notebooks 02 and 03 produced

If they ran in this same session the files are already on disk. If not, upload
the zips those notebooks downloaded and this cell unpacks them in place.

In [ ]:
import json
import zipfile
from pathlib import Path

WANT = ["data/results/antispoof_branch_scores.json",
        "data/results/intent_branch_scores.json",
        "data/models/antispoof_rawnet.pt"]

missing = [w for w in WANT if not Path(w).exists()]
print("already here:", [w for w in WANT if w not in missing])
print("missing:     ", missing)

if missing:
    try:
        from google.colab import files

        print("\nUpload the zips from notebooks 02 and 03 (or press cancel to skip).")
        uploaded = files.upload()
        for name in uploaded:
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(name) as z:
                    z.extractall(".")
                print(f"unpacked {name}")
    except Exception as exc:
        print(f"upload skipped ({type(exc).__name__}). "
              f"Falling back to the locally trained branches.")

HAVE_AUDIO_BRANCH = Path("data/results/antispoof_branch_scores.json").exists()
HAVE_TEXT_BRANCH = Path("data/results/intent_branch_scores.json").exists()
HAVE_RAWNET = Path("data/models/antispoof_rawnet.pt").exists()
print(f"\nreal-data audio branch: {HAVE_AUDIO_BRANCH}")
print(f"real-data text branch:  {HAVE_TEXT_BRANCH}")
print(f"rawnet checkpoint:      {HAVE_RAWNET}")

In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch

from swarkavach import config

config.ensure_dirs()          # data/results must exist before the first savefig

HAS_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
print(f"device {DEVICE}")

if not any(Path("data/corpus/calls").glob("*.json")):
    print("no corpus, generating one")
    subprocess.run([sys.executable, "-m", "swarkavach.cli", "gen-corpus",
                    "--n", "480", "--audio"], check=False)

if not Path("data/models/fusion_full_logreg.joblib").exists():
    print("no trained local models, running swarkavach train")
    proc = subprocess.run([sys.executable, "-m", "swarkavach.cli", "train"],
                          capture_output=True, text=True)
    print((proc.stdout or "")[-2500:])
    if proc.returncode != 0:
        print((proc.stderr or "")[-2500:])

from swarkavach.pipeline import Pipeline

P = Pipeline.load()
print(json.dumps(P.status(), indent=2))
CALLS = P.calls
print(f"\n{len(CALLS)} calls, cells {dict(Counter(c.cell for c in CALLS))}")

## Rebuild the 25-feature vectors with real-data branch scores

`fusion.featurize.build_features` fills all 25 slots from the corpus and the
locally trained models. Two of those groups are then overwritten with the
numbers the real-data models produced:

- `as_score` from the ASVspoof-trained RawNet checkpoint, plus `as_llr` as its
  log-odds clipped to the fusion range and `as_margin` as `2 * p - 1`, which
  is the scale the rest of the voice group is written on
- `intent_score` and `intent_max_turn` from the fine-tuned MuRIL

Everything else, the entity counts, the code-mixing profile, the coercion
trajectory and the prosody-intent mismatch, is computed here from the corpus,
because those features are the project's own and there is no external model to
replace them with.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np

from swarkavach.audioio import read_audio
from swarkavach.config import SETTINGS, TARGET_SR
from swarkavach.fusion.featurize import build_features
from swarkavach.schema import FUSION_FEATURE_NAMES, feature_vector

AUDIO_BRANCH = {}
TEXT_BRANCH = {}
if HAVE_AUDIO_BRANCH:
    AUDIO_BRANCH = json.loads(
        Path("data/results/antispoof_branch_scores.json").read_text(encoding="utf-8")
    ).get("scores", {})
if HAVE_TEXT_BRANCH:
    TEXT_BRANCH = json.loads(
        Path("data/results/intent_branch_scores.json").read_text(encoding="utf-8")
    ).get("scores", {})
print(f"{len(AUDIO_BRANCH)} audio branch scores, {len(TEXT_BRANCH)} text branch scores")


def logit(p, lo=1e-4):
    p = float(np.clip(p, lo, 1 - lo))
    return float(np.clip(np.log(p / (1 - p)), -10.0, 10.0))


def featurise(calls, with_audio=True, override=True, audio_override=None):
    """(X, y, feature dicts, calls kept), with the real-data branches folded in."""
    audio_override = AUDIO_BRANCH if audio_override is None else audio_override
    X, y, feats, kept = [], [], [], []
    t0 = time.time()
    for i, c in enumerate(calls):
        audio = sr = None
        if with_audio and c.audio_path and Path(c.audio_path).exists():
            try:
                audio, sr = read_audio(c.audio_path, sr=SETTINGS.frame.sr)
            except Exception:
                audio = sr = None
        f, _ = build_features(c, audio=audio, sr=sr, models=P.models)

        if override:
            a = audio_override.get(c.call_id)
            if a is not None:
                p = float(a["as_score"] if isinstance(a, dict) else a)
                f["as_score"] = p
                f["as_llr"] = logit(p)
                f["as_margin"] = 2.0 * p - 1.0
            t = TEXT_BRANCH.get(c.call_id)
            if t is not None:
                f["intent_score"] = float(t)
                f["intent_max_turn"] = float(max(float(t), f.get("intent_max_turn", 0.0)))

        X.append(feature_vector(f))
        y.append(int(c.label_scam))
        feats.append(f)
        kept.append(c)
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(calls)} ({time.time() - t0:.0f}s)", flush=True)
    return (np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.int64), feats, kept)


TRAIN = [c for c in CALLS if c.split == "train"]
TEST = [c for c in CALLS if c.split == "test"] or CALLS
print(f"\nfeaturising {len(TRAIN)} train calls")
Xtr, ytr, ftr, ctr = featurise(TRAIN)
print(f"featurising {len(TEST)} test calls")
Xte, yte, fte, cte = featurise(TEST)
print(f"\nX train {Xtr.shape}, X test {Xte.shape}, scam share "
      f"{ytr.mean():.2f} / {yte.mean():.2f}")

REAL_DATA_BRANCHES = {
    "audio": bool(AUDIO_BRANCH),
    "text": bool(TEXT_BRANCH),
}
print("real-data branch overrides in use:", REAL_DATA_BRANCHES)

## The ablation table

Four models, one per arm, trained on the same rows with the same seed. The
per-cell number is the fraction of calls in that cell the arm got right at the
alert threshold, which is 0.65 by default and comes from
`SETTINGS.pipeline.alert_threshold`.

Read the two off-diagonal columns first. `humanxscam` is where the audio
branch has nothing to say, `clonedxbenign` is where it says the wrong thing
loudly. If fusion is worth anything it shows up there before it shows up in
the aggregate AUC.

In [ ]:
import numpy as np
import pandas as pd

from swarkavach.evaluate import binary_scores, eer, roc_auc
from swarkavach.fusion.model import train_all_arms
from swarkavach.schema import ABLATION_ARMS

THRESHOLD = SETTINGS.pipeline.alert_threshold
CELLS = ["humanxbenign", "humanxscam", "clonedxbenign", "clonedxscam"]

for arm, names in ABLATION_ARMS.items():
    print(f"{arm:<12} {len(names):>2} features")

ARMS = train_all_arms(Xtr, ytr, kind="logreg")
if "full" not in ARMS:
    raise RuntimeError("the full arm failed to train, everything below depends on it")
ARM_SCORES, OVERALL, DET_RATE = {}, {}, {}

for arm, model in ARMS.items():
    s = model.predict_proba(Xte)
    ARM_SCORES[arm] = s
    b = binary_scores(s, yte, THRESHOLD)
    e, _ = eer(s, yte)
    OVERALL[arm] = {"auc": round(roc_auc(s, yte), 4),
                    "accuracy": round(b["accuracy"], 4),
                    "f1": round(b["f1"], 4),
                    "precision": round(b["precision"], 4),
                    "recall": round(b["recall"], 4),
                    "eer": round(e, 4)}
    per_cell = {}
    for cell in CELLS:
        idx = [i for i, c in enumerate(cte) if c.cell == cell]
        if not idx:
            per_cell[cell] = None
            continue
        want = yte[idx]
        got = (s[idx] >= THRESHOLD).astype(int)
        per_cell[cell] = round(float((got == want).mean()), 4)
    DET_RATE[arm] = per_cell

N_PER_CELL = {c: int(sum(1 for x in cte if x.cell == c)) for c in CELLS}

table = pd.DataFrame(
    [[DET_RATE[a][c] for c in CELLS] + [OVERALL[a]["auc"], OVERALL[a]["accuracy"],
                                        OVERALL[a]["f1"], OVERALL[a]["eer"]]
     for a in ARMS],
    index=list(ARMS),
    columns=[f"{c} (n={N_PER_CELL[c]})" for c in CELLS] + ["AUC", "accuracy", "F1", "EER"])

print(f"\nthreshold {THRESHOLD}, {len(cte)} held-out calls\n")
print(table.to_string())

full = OVERALL.get("full", {}).get("auc")
late = OVERALL.get("late_fusion", {}).get("auc")
best_single = max(OVERALL.get("audio_only", {}).get("auc", 0),
                  OVERALL.get("text_only", {}).get("auc", 0))
print(f"\nfull {full}   late_fusion {late}   best single branch {best_single}")
if full is not None and late is not None:
    print(f"cross-modal block is worth {full - late:+.4f} AUC over late fusion")
    print(f"fusion is worth {full - best_single:+.4f} AUC over the best single branch")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

arms = list(ARMS)
labels = {"audio_only": "Audio only", "text_only": "Text only",
          "late_fusion": "Late fusion", "full": "Full (cross-modal)"}
cell_labels = {"humanxbenign": "Human /\nbenign", "humanxscam": "Human /\nscam",
               "clonedxbenign": "Cloned /\nbenign", "clonedxscam": "Cloned /\nscam"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4),
                               gridspec_kw={"width_ratios": [1.5, 1]})

x = np.arange(len(CELLS))
w = 0.8 / len(arms)
for i, a in enumerate(arms):
    vals = [DET_RATE[a][c] or 0.0 for c in CELLS]
    bars = ax1.bar(x + (i - (len(arms) - 1) / 2) * w, vals, w, label=labels.get(a, a))
    for b, v in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.2f}",
                 ha="center", fontsize=7.5)
ax1.set_xticks(x)
ax1.set_xticklabels([cell_labels[c] for c in CELLS])
ax1.set_ylim(0, 1.09)
ax1.set_ylabel("correct at the alert threshold")
ax1.set_title("The four cells. Watch the two middle columns.")
ax1.legend(fontsize=8, ncol=2)
ax1.grid(axis="y", alpha=0.3)

grid = np.array([[DET_RATE[a][c] if DET_RATE[a][c] is not None else np.nan
                  for c in CELLS] for a in arms])
im = ax2.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax2.set_xticks(range(len(CELLS)))
ax2.set_xticklabels([c.replace("x", "\n") for c in CELLS], fontsize=8)
ax2.set_yticks(range(len(arms)))
ax2.set_yticklabels([labels.get(a, a) for a in arms], fontsize=9)
for i in range(len(arms)):
    for j in range(len(CELLS)):
        if np.isfinite(grid[i, j]):
            ax2.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax2.set_title("same numbers, as a grid")
fig.colorbar(im, ax=ax2, fraction=0.045)

plt.tight_layout()
plt.savefig("data/results/ablation_grid.png", dpi=140)
plt.show()

## Calibration

Plan B section 5 says to check calibration first when fusion underperforms,
and the reason is mechanical: combining two uncalibrated scores produces a
fusion that follows whichever branch happens to be more confident rather than
whichever is more correct. So this curve is a precondition for the table
above, not a footnote to it.

Brier score is the mean squared error of the probabilities. Expected
calibration error is the average gap between predicted and observed
probability, weighted by bin occupancy.

In [ ]:
import matplotlib.pyplot as plt

from swarkavach.evaluate import calibration_curve

CAL = calibration_curve(ARM_SCORES["full"], yte, n_bins=10)
print(f"Brier {CAL['brier']}   ECE {CAL['ece']}   {len(CAL['bins'])} occupied bins")
for b in CAL["bins"]:
    print(f"  predicted {b['p_pred']:.3f}  observed {b['p_obs']:.3f}  n={b['n']}")

fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], "k--", lw=0.9, label="perfect")
ax.plot([b["p_pred"] for b in CAL["bins"]], [b["p_obs"] for b in CAL["bins"]],
        marker="o", label="full arm")
ax.set_xlabel("predicted fraud probability")
ax.set_ylabel("observed fraud rate")
ax.set_title(f"Calibration, Brier {CAL['brier']:.4f}, ECE {CAL['ece']:.4f}")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig("data/results/calibration.png", dpi=140)
plt.show()

## Codec robustness

Real calls arrive through a codec. G.711 mu-law and A-law are implemented
exactly in numpy in `swarkavach.channel`, so those two conditions are real
regardless of what is installed. GSM 06.10 and AMR-NB need ffmpeg; without it
the module falls back to a documented numpy approximation and sets
`real_codec: False`, which the export carries and the console shows in a
different colour. Not hiding that flag is the difference between a result and
a claim.

The whole feature vector is rebuilt on the degraded audio, not just the audio
branch, because the prosody-intent mismatch reads pitch and energy off the
same waveform and a codec moves both.

In [ ]:
import numpy as np

from swarkavach.channel import CODECS, degrade
from swarkavach.config import find_ffmpeg

FFMPEG = find_ffmpeg()
CODEC_LIST = ["clean", "g711u", "g711a", "narrowband", "packet_loss"]
if FFMPEG:
    CODEC_LIST += ["gsm", "amrnb"]
else:
    print("no ffmpeg on this runtime: GSM and AMR-NB would be numpy approximations.")
    print("Colab usually has ffmpeg; if this prints, run  !apt-get -qq install ffmpeg")

RAWNET = None
if HAVE_RAWNET:
    from swarkavach.antispoof.rawnet import load_rawnet

    RAWNET = load_rawnet("data/models/antispoof_rawnet.pt")
    if RAWNET is not None:
        RAWNET = RAWNET.to(DEVICE).eval()
        print("re-scoring degraded audio with the ASVspoof-trained RawNet")
    else:
        print("checkpoint did not load, the voice group keeps its clean-audio score")


def rawnet_score(x, sr):
    """P(spoof) for one waveform, averaged over 4 second segments."""
    from swarkavach.audioio import pad_or_trim, rms_normalize

    n = int(RAWNET.n_samples)
    sig = rms_normalize(np.asarray(x, dtype=np.float32))
    segs = [pad_or_trim(sig[s:s + n], n) for s in range(0, max(len(sig) - n, 0) + 1, n)]
    if not segs:
        segs = [pad_or_trim(sig, n)]
    with torch.no_grad():
        xb = torch.from_numpy(np.stack(segs).astype(np.float32)).to(DEVICE)
        p = torch.softmax(RAWNET(xb), dim=1)[:, 1].float().cpu().numpy()
    return float(np.mean(p))


FULL_MODEL = ARMS["full"]
WITH_AUDIO = [c for c in cte if c.audio_path and Path(c.audio_path).exists()]
y_rb = np.asarray([c.label_scam for c in WITH_AUDIO], dtype=int)
print(f"{len(WITH_AUDIO)} test calls have audio")

ROBUST = {"codecs": [], "auc": {}, "eer": {}, "real_codec": {}, "labels": {}}
for codec in CODEC_LIST:
    scores, is_real = [], True
    t0 = time.time()
    for c in WITH_AUDIO:
        try:
            x, sr = read_audio(c.audio_path, sr=SETTINGS.frame.sr)
            if codec != "clean":
                x, info = degrade(x, sr, codec=codec, seed=SETTINGS.pipeline.seed)
                is_real = bool(info.get("real_codec", True)) and is_real
            f, _ = build_features(c, audio=x, sr=sr, models=P.models)
            if RAWNET is not None:
                p = rawnet_score(x, sr)
                f["as_score"] = p
                f["as_llr"] = logit(p)
                f["as_margin"] = 2.0 * p - 1.0
            t = TEXT_BRANCH.get(c.call_id)
            if t is not None:
                f["intent_score"] = float(t)
            scores.append(float(FULL_MODEL.predict_one(f)))
        except Exception:
            scores.append(0.0)
    s = np.asarray(scores)
    ROBUST["codecs"].append(codec)
    ROBUST["auc"][codec] = round(roc_auc(s, y_rb), 4)
    ROBUST["eer"][codec] = round(eer(s, y_rb)[0], 4)
    ROBUST["real_codec"][codec] = bool(is_real)
    ROBUST["labels"][codec] = CODECS.get(codec, {}).get("label", codec)
    print(f"{codec:<12} AUC {ROBUST['auc'][codec]:.4f}  EER {ROBUST['eer'][codec]:.4f}  "
          f"{'real codec' if is_real else 'numpy approximation'}  "
          f"({time.time() - t0:.0f}s)")

ROBUST["n"] = len(WITH_AUDIO)
ROBUST["ffmpeg"] = bool(FFMPEG)

In [ ]:
import matplotlib.pyplot as plt

codecs = ROBUST["codecs"]
vals = [ROBUST["auc"][c] for c in codecs]
colors = ["tab:blue" if ROBUST["real_codec"][c] else "tab:orange" for c in codecs]

fig, ax = plt.subplots(figsize=(7.6, 3.6))
bars = ax.bar(range(len(codecs)), vals, color=colors)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008, f"{v:.3f}", ha="center", fontsize=8)
ax.set_xticks(range(len(codecs)))
ax.set_xticklabels([ROBUST["labels"][c] for c in codecs], rotation=25, ha="right", fontsize=8)
ax.set_ylabel("end-to-end AUC")
ax.set_ylim(0, 1.05)
ax.set_title("Fused AUC through each channel condition (orange = numpy approximation)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("data/results/robustness.png", dpi=140)
plt.show()

## Time to detection

Accuracy on a finished recording is the wrong question for a system meant to
interrupt a live call. `fusion.streaming.stream_call` re-scores after every
turn on the prefix heard so far, and the first crossing of the alert threshold
is the time to detection.

The fusion model used for streaming is set to the `full` arm trained above, so
this measures the same system the table measures. Only scam calls are counted:
time to detection on a benign call is not a thing.

In [ ]:
import numpy as np

from swarkavach.fusion.streaming import stream_call, time_to_detection, ttd_stats

P.models.fusion = ARMS["full"]

records = []
t0 = time.time()
scam_calls = [c for c in cte if c.label_scam]
for k, c in enumerate(scam_calls):
    audio = sr = None
    if c.audio_path and Path(c.audio_path).exists():
        try:
            audio, sr = read_audio(c.audio_path, sr=SETTINGS.frame.sr)
        except Exception:
            pass
    tl = list(stream_call(c, audio=audio, sr=sr, models=P.models, threshold=THRESHOLD))
    t, turn = time_to_detection(tl, THRESHOLD)
    records.append({"call_id": c.call_id, "scenario": c.scenario,
                    "ttd": t, "ttd_turns": turn, "duration": c.duration})
    if (k + 1) % 25 == 0:
        print(f"  {k + 1}/{len(scam_calls)} ({time.time() - t0:.0f}s)", flush=True)

TTD = ttd_stats(records)
TTD["by_scenario"] = {}
for r in records:
    TTD["by_scenario"].setdefault(r["scenario"], []).append(r["ttd"])
TTD["by_scenario"] = {
    k: round(float(np.median([x for x in v if x is not None])), 2)
    for k, v in TTD["by_scenario"].items() if any(x is not None for x in v)}
TTD["threshold"] = THRESHOLD

print(f"\ndetected {TTD['detected']} of {TTD['total']} scam calls "
      f"({TTD['detection_rate'] * 100:.0f}%)")
for k in ("median", "mean", "p90", "min", "max", "median_turns"):
    if k in TTD:
        print(f"  {k:<13} {TTD[k]}")
print("\nmedian seconds by scenario:")
for k, v in sorted(TTD["by_scenario"].items(), key=lambda kv: kv[1]):
    print(f"  {k:<24} {v}")

In [ ]:
import matplotlib.pyplot as plt

if TTD.get("values"):
    fig, ax = plt.subplots(figsize=(6.6, 3.6))
    ax.hist(TTD["values"], bins=14, color="tab:green", alpha=0.85, edgecolor="white")
    if "median" in TTD:
        ax.axvline(TTD["median"], color="tab:orange", ls="--",
                   label=f"median {TTD['median']:.1f}s")
        ax.legend()
    ax.set_xlabel("seconds of call heard before the alert")
    ax.set_ylabel("calls")
    ax.set_title(f"Time to detection, threshold {THRESHOLD}, "
                 f"{TTD['detected']} of {TTD['total']} scam calls flagged")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("data/results/ttd.png", dpi=140)
    plt.show()
else:
    print("no call crossed the threshold, nothing to plot")

## Export

Four files, in the exact shapes `evaluate.py` writes, so the console renders
them with no frontend change. `provenance.json` is what tells a reader that
these particular numbers came from a GPU run with real-data branch scores
rather than from the laptop.

In [ ]:
# ---------------------------------------------------------------------------
# Export helpers. Each notebook carries its own copy so it can run on its own.
#
# Everything written here lands in data/results/ with "source": "colab", a UTC
# timestamp and the GPU name, which is what the local console's provenance
# table reads to tell your laptop numbers apart from your Colab numbers.
# ---------------------------------------------------------------------------
import json
import platform
import zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK = "04_fusion_and_eval"
RESULTS = Path("data/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "none (CPU runtime)"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def colab_stamp(notes: str = "") -> dict:
    try:
        import torch

        tv = str(torch.__version__)
    except Exception:
        tv = "not installed"
    return {
        "source": "colab",
        "generated": utc_now(),
        "gpu": gpu_name(),
        "torch": tv,
        "python": platform.python_version(),
        "notebook": NOTEBOOK,
        "notes": notes,
    }


WRITTEN = []


def write_result(name: str, obj: dict, notes: str = "") -> Path:
    """Write one result file, stamped, and remember it for the zip."""
    doc = dict(obj)
    doc.update(colab_stamp(notes))
    path = RESULTS / name
    path.write_text(json.dumps(doc, indent=2, default=float), encoding="utf-8")
    if name not in WRITTEN:
        WRITTEN.append(name)
    print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} kB)")
    return path


def update_provenance(entries) -> Path:
    """Merge these Colab entries into data/results/provenance.json.

    Entries for other artifacts are kept, so a local `swarkavach evaluate` run
    and a Colab run can sit in the same table. Unzip the Colab results AFTER
    the local evaluate, otherwise the local file overwrites this one.
    """
    path = RESULTS / "provenance.json"
    doc = {"entries": []}
    if path.exists():
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    new_names = {e["artifact"] for e in entries}
    kept = [e for e in doc.get("entries", []) if e.get("artifact") not in new_names]
    doc["entries"] = kept + list(entries)
    doc["generated"] = utc_now()
    doc["host"] = platform.platform()
    doc["gpu"] = gpu_name()
    path.write_text(json.dumps(doc, indent=2), encoding="utf-8")
    if "provenance.json" not in WRITTEN:
        WRITTEN.append("provenance.json")
    print(f"provenance now lists {len(doc['entries'])} artifacts")
    return path


def export_zip(extra_paths=(), zip_name: str = None) -> Path:
    """Zip the result files plus anything extra, then hand it to the browser."""
    out = Path(zip_name or f"swarkavach_{NOTEBOOK}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in WRITTEN:
            f = RESULTS / name
            if f.exists():
                z.write(f, arcname=f"data/results/{f.name}")
        for extra in extra_paths:
            p = Path(extra)
            if p.exists():
                z.write(p, arcname=p.as_posix())
            else:
                print(f"  skipped missing {p}")

    names = zipfile.ZipFile(out).namelist()
    print(f"\n{out.name}  {out.stat().st_size / 1024:.1f} kB")
    for n in names:
        print("   ", n)

    try:
        from google.colab import files

        files.download(str(out))
        print("\nDownload started. Unzip it over your local project root.")
    except Exception as exc:
        print(f"\nnot running on Colab ({type(exc).__name__}), zip is at {out.resolve()}")
    return out

In [ ]:
from pathlib import Path

ablation = {
    "arms": list(ARMS),
    "cells": CELLS,
    "detection_rate": DET_RATE,
    "overall": OVERALL,
    "n_per_cell": N_PER_CELL,
    "threshold": THRESHOLD,
    "split": "test",
    "n_test": len(cte),
    "n_train": len(ctr),
    "real_data_branches": REAL_DATA_BRANCHES,
    "feature_counts": {a: len(n) for a, n in ABLATION_ARMS.items()},
    "note": ("audio branch scores come from the ASVspoof-trained RawNet checkpoint "
             "when notebook 02 has run, text branch scores from the fine-tuned MuRIL "
             "when notebook 03 has run. The cross-modal block is computed here from "
             "the corpus in both cases."),
}
write_result("ablation_results.json", ablation,
             notes=f"four arms on {len(cte)} held-out calls, threshold {THRESHOLD}")

write_result("calibration.json", CAL,
             notes="10 equal-width bins on the test split, full arm")

write_result("robustness_results.json", ROBUST,
             notes=f"{len(CODEC_LIST)} channel conditions, ffmpeg={bool(FFMPEG)}")

write_result("ttd_results.json", TTD,
             notes=f"scam calls in the test split, alert threshold {THRESHOLD}")

branch_note = ("real-data audio and text branches"
               if all(REAL_DATA_BRANCHES.values())
               else f"branch overrides: {REAL_DATA_BRANCHES}")
update_provenance([
    {"artifact": "ablation_results.json", "source": "colab", "generated": utc_now(),
     "notes": f"{len(cte)} held-out calls, threshold {THRESHOLD}, {branch_note}"},
    {"artifact": "calibration.json", "source": "colab", "generated": utc_now(),
     "notes": f"full arm, Brier {CAL['brier']}, ECE {CAL['ece']}"},
    {"artifact": "robustness_results.json", "source": "colab", "generated": utc_now(),
     "notes": f"{', '.join(CODEC_LIST)}; real codecs where ffmpeg exists"},
    {"artifact": "ttd_results.json", "source": "colab", "generated": utc_now(),
     "notes": f"{TTD['detected']} of {TTD['total']} scam calls flagged"},
])

export_zip(extra_paths=[Path("data/results/ablation_grid.png"),
                        Path("data/results/calibration.png"),
                        Path("data/results/robustness.png"),
                        Path("data/results/ttd.png")])

## How to read the table in the report

Three sentences, in this order.

1. What the aggregate AUC of `full` is, and what the best single branch got.
2. Which cell the gain came from. If `full` beats `audio_only` mostly on
   `humanxscam`, that is the audio branch being blind to a human scammer, and
   it is the expected result. If it beats `text_only` mostly on
   `clonedxbenign`, look again, because that cell is where a text-only model
   should already be right and an audio-only model should be wrong.
3. What `full` minus `late_fusion` is. That difference is the only thing in
   this table that is attributable to the cross-modal features, and if it is
   inside the noise the honest write-up says so and falls back to the
   complementarity argument in Plan B section 5.3.